# 시나리오 모델 파인튜닝 (Unsloth + QLoRA)

SKT A.X 3.1 Light (7B) 모델을 밈 시나리오 생성에 특화되도록 파인튜닝

## 환경
- Colab T4 (무료) 또는 A100
- Unsloth (2x faster, 60% less memory)
- QLoRA 4bit 양자화

## 1. 설치

In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

## 2. 모델 로드

In [ ]:
from unsloth import FastLanguageModel

# SKT A.X 3.1 Light 7B (한국어 특화, CLIcK 71.22)
# HuggingFace: https://huggingface.co/SKT/A.X-3.1-7B-Instruct-Light
MODEL_NAME = "SKT/A.X-3.1-7B-Instruct-Light"

# 대체 모델 (SKT 모델 접근 불가 시)
# MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"  # Qwen 기반 (중국어 섞임 가능)
# MODEL_NAME = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"  # LG EXAONE

max_seq_length = 4096
dtype = None  # Auto detect
load_in_4bit = True  # QLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

## 3. LoRA 설정

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,  # Unsloth 최적화
    bias="none",
    use_gradient_checkpointing="unsloth",  # 메모리 최적화
    random_state=42,
)

# 학습 가능 파라미터 수 확인
model.print_trainable_parameters()

## 4. 데이터 로드

### 옵션 A: Google Drive에서 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Google Drive 경로 설정
DATA_PATH = "/content/drive/MyDrive/meme-fluencer/finetune/data/splits"

# 또는 직접 업로드
# from google.colab import files
# uploaded = files.upload()  # train.jsonl, val.jsonl 업로드

In [ ]:
import json
from datasets import Dataset

def load_jsonl(path):
    data = []
    with open(path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return data

# 데이터 로드
train_data = load_jsonl(f"{DATA_PATH}/train.jsonl")
val_data = load_jsonl(f"{DATA_PATH}/val.jsonl")

print(f"Train: {len(train_data)} samples")
print(f"Val: {len(val_data)} samples")

### 옵션 B: HuggingFace Hub에서 로드

In [ ]:
# HuggingFace Hub에 데이터셋 업로드한 경우
# from datasets import load_dataset
# dataset = load_dataset("your-username/meme-scenario-dataset")
# train_data = dataset["train"]
# val_data = dataset["validation"]

## 5. 데이터 포맷팅

In [ ]:
# Chat 템플릿 적용
def format_chat(example):
    messages = example["messages"]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

# Dataset 변환
train_dataset = Dataset.from_list(train_data).map(format_chat)
val_dataset = Dataset.from_list(val_data).map(format_chat)

print("Sample formatted text:")
print(train_dataset[0]["text"][:500] + "...")

## 6. 학습 설정

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # 짧은 시퀀스 패킹 (메모리 절약)
    args=TrainingArguments(
        # 출력
        output_dir="./outputs",
        
        # 배치 크기 (T4: 2, A100: 8)
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,  # effective batch = 8
        
        # 학습률
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        
        # 에폭
        num_train_epochs=3,
        
        # 정밀도
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        
        # 로깅
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        
        # 최적화
        optim="adamw_8bit",
        weight_decay=0.01,
        max_grad_norm=1.0,
        
        # 기타
        seed=42,
        report_to="none",  # wandb 사용 시 "wandb"
    ),
)

## 7. 학습 실행

In [ ]:
# GPU 메모리 확인
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"Max memory: {max_memory} GB")
print(f"Reserved: {start_gpu_memory} GB")

In [ ]:
# 학습 시작
trainer_stats = trainer.train()

In [ ]:
# 학습 결과
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f}s")
print(f"Peak GPU memory: {used_memory} GB")
print(f"Final train loss: {trainer_stats.metrics['train_loss']:.4f}")

## 8. 추론 테스트

In [ ]:
# 추론 모드로 전환
FastLanguageModel.for_inference(model)

# 테스트 프롬프트
test_messages = [
    {
        "role": "system",
        "content": "당신은 밈을 활용한 숏폼 광고 시나리오 전문가입니다. 주어진 밈과 제품 정보를 바탕으로 재미있고 바이럴 가능한 시나리오를 작성합니다."
    },
    {
        "role": "user",
        "content": """## 밈 정보
- 밈 이름: 무야호
- 정의: 기쁨이나 흥분을 표현할 때 쓰는 감탄사
- 핵심 대사: 무야호~!

## 제품 정보
- 회사: 테스트컴퍼니
- 제품: 에너지 드링크
- 카테고리: 음료
- 키메시지: 에너지 충전

## 요청
위 밈과 제품을 활용한 15-30초 숏폼 광고 시나리오를 작성해주세요.
톤: 유쾌
"""
    }
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=1024,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

## 9. 모델 저장

In [ ]:
# LoRA 어댑터만 저장 (작은 용량)
model.save_pretrained("./lora_adapter")
tokenizer.save_pretrained("./lora_adapter")

print("LoRA adapter saved to ./lora_adapter")

In [ ]:
# Google Drive에 백업
!cp -r ./lora_adapter /content/drive/MyDrive/meme-fluencer/models/scenario_lora/

### HuggingFace Hub 업로드 (선택)

In [ ]:
# HuggingFace 로그인
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")  # https://huggingface.co/settings/tokens

# LoRA 어댑터 업로드
model.push_to_hub("your-username/meme-scenario-lora", token="YOUR_HF_TOKEN")
tokenizer.push_to_hub("your-username/meme-scenario-lora", token="YOUR_HF_TOKEN")

## 10. 병합 모델 저장 (GGUF 변환용)

In [ ]:
# LoRA를 베이스 모델에 병합 (16bit)
model.save_pretrained_merged(
    "./merged_model",
    tokenizer,
    save_method="merged_16bit",
)

print("Merged model saved to ./merged_model")

In [ ]:
# GGUF 변환 (llama.cpp 호환, Ollama 사용 가능)
model.save_pretrained_gguf(
    "./gguf_model",
    tokenizer,
    quantization_method="q4_k_m",  # 4bit 양자화
)

print("GGUF model saved to ./gguf_model")

## 완료!

### 다음 단계
1. **LoRA 어댑터**: 서빙 시 베이스 모델 + LoRA 로드
2. **GGUF**: Ollama나 llama.cpp로 로컬 서빙
3. **HuggingFace**: 클라우드 API 서빙

### 사용 예시 (Python)
```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "your-username/meme-scenario-lora",
    max_seq_length=4096,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
```